# What AI Drinks

Day 20 · Hexagons. The water basins under America's AI campuses: how stressed the rivers are (WRI Aqueduct
baseline water stress) and whether the groundwater underneath is dropping (Aqueduct groundwater table decline).
These are the basins the campuses sit in, not their exact intake pipes. Run `python scripts\fetch_week3.py day20` first.

## 1 · Settings

In [ ]:
DAY      = 20
PROMPT   = "Hexagons"
STUDY    = "B"
SLUG     = "hexagons-what-ai-drinks"
NOTEBOOK = "day20_hexagons_what_ai_drinks.ipynb"
TITLE    = "What AI Drinks"
HIGHLIGHT = "AI"
CREDITS  = ("Water stress and groundwater decline: WRI Aqueduct 4.0 (CC BY 4.0). AI campuses: Epoch AI, Frontier Data "
            "Centers (CC BY 4.0). Existing data centers: IM3 Open Source Data Center Atlas (PNNL), built on © OpenStreetMap "
            "(ODbL). Boundaries: US Census Bureau.")

HEX_KM   = 70           # hexagon width, flat to flat
STRESSED = 3            # Aqueduct category 3 = high, 4 = extremely high
DROPPING = 3            # groundwater decline category 3 or 4 = high or extremely high

PAGE, INK, SOFT, LAND = "#0e1418", "#eef3f4", "#8fa0a6", "#1a2329"
WATER = {-9999: "#20292e", -1: "#2a3338", 0: "#163845", 1: "#1d5566", 2: "#2b7d8c", 3: "#5fb3b3", 4: "#b8eee6"}   # Aqueduct bws_cat, dark = calm, bright = stressed
WATER_LABEL = {-9999: "No data", 0: "Low", 1: "Low-medium", 2: "Medium-high", 3: "High", 4: "Extremely high", -1: "Arid, low use"}
RUNNING, COMING, PINK = "#f2b134", "#ff5c8a", "#ff5c8a"

## 2 · Load

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from mapkit import ai, poster, publish
from mapkit.paths import PROCESSED, EQUAL_AREA

basins = gpd.read_file(PROCESSED / "aqueduct_us_basins.geojson").to_crs(EQUAL_AREA)
master = gpd.read_file(PROCESSED / "datacenters_master.geojson").to_crs(EQUAL_AREA)
states = ai.load_states()
states = states[~states.state_abb.isin(["AK", "HI", "PR", "VI", "GU", "AS", "MP"])].to_crs(EQUAL_AREA)
campus = master[master.kind == "ai_campus"].copy()
for c in ("bws_cat", "gtd_cat"):
    if c in basins:
        basins[c] = pd.to_numeric(basins[c], errors="coerce")
print(f"{len(basins):,} basins · {len(campus)} AI campuses · columns: {[c for c in basins.columns if c != 'geometry']}")

## 3 · Analysis, every number on the map comes from here

In [ ]:
# Each campus takes the values of the basin it sits in
j = gpd.sjoin(campus, basins[["bws_cat", "gtd_cat", "geometry"]], how="left", predicate="within")
j = j[~j.index.duplicated()]
campus["basin_bws"] = j["bws_cat_right"] if "bws_cat_right" in j else j["bws_cat"]
campus["basin_gtd"] = j["gtd_cat"]
# Water stress per campus comes from the master table: it was joined on Aqueduct's full-detail basins on Day 1.
# (The basins file here is simplified for drawing, which can nudge a campus sitting on a boundary.)
campus["stressed"] = pd.to_numeric(campus.bws_cat, errors="coerce") >= STRESSED
campus["dropping"] = campus.basin_gtd >= DROPPING
campus["running"] = campus.status == "Operating"

N = len(campus)
STATS = dict(
    stressed=campus.stressed.sum(), dropping=campus.dropping.sum(), both=(campus.stressed & campus.dropping).sum(),
    peak_mw=campus.power_peak_mw.sum(),
    peak_mw_stressed=campus.power_peak_mw[campus.stressed].sum(),
)
print(f"{STATS['stressed']} of {N} AI campuses ({STATS['stressed']/N:.0%}) sit in high or extremely high water stress")
print(f"{STATS['dropping']} of {N} ({STATS['dropping']/N:.0%}) sit over groundwater that is dropping fast (high or extremely high decline)")
print(f"{STATS['both']} face both")
print(f"Power at full build in stressed basins: {STATS['peak_mw_stressed']/1000:.1f} of {STATS['peak_mw']/1000:.1f} GW "
      f"({STATS['peak_mw_stressed']/STATS['peak_mw']:.0%})")
print(campus.groupby("status")[["stressed", "dropping"]].mean().round(2))

# Hexagons: each hex takes the water stress of the basin covering most of it
grid = ai.hexbin(gpd.GeoDataFrame(geometry=states.representative_point(), crs=EQUAL_AREA), size_km=HEX_KM,
                 bounds_gdf=states, keep_empty=True).to_crs(EQUAL_AREA)
grid["hid"] = range(len(grid))
ov = gpd.overlay(grid[["hid", "geometry"]], basins[["bws_cat", "geometry"]], how="intersection", keep_geom_type=True)
ov["a"] = ov.area
grid["bws"] = grid.hid.map(ov.sort_values("a").groupby("hid").bws_cat.last())
grid = grid.dropna(subset=["bws"])
grid = grid[grid.intersects(states.union_all())]
print(f"{len(grid):,} hexagons")

## 4 · Map

In [ ]:
F = poster.font
fig, ax = poster.page(PAGE)
ax.set_position([0.02, 0.215, 0.96, 0.60])
for cat, col in WATER.items():
    grid[grid.bws == cat].plot(ax=ax, color=col, edgecolor=PAGE, lw=0.8, zorder=1)
states.boundary.plot(ax=ax, color=PAGE, lw=0.5, alpha=0.6, zorder=2)

size = lambda mw: 14 + np.sqrt(mw.fillna(0)) * 5
for running, col in ((True, RUNNING), (False, COMING)):
    c = campus[campus.running == running]
    ax.scatter(c.geometry.x, c.geometry.y, s=size(c.power_peak_mw) * 3, color=col, alpha=0.18, lw=0, zorder=4)
    ax.scatter(c.geometry.x, c.geometry.y, s=size(c.power_peak_mw), color=col, edgecolor=PAGE, lw=0.6, zorder=5)
d = campus[campus.dropping]
ax.scatter(d.geometry.x, d.geometry.y, s=size(d.power_peak_mw) * 4, facecolor="none", edgecolor=INK, lw=1.6,
           ls=(0, (2.5, 1.5)), zorder=6)
b = states.total_bounds
ax.set_xlim(b[0], b[2]); ax.set_ylim(b[1], b[3]); ax.set_aspect("equal"); ax.set_axis_off()

# title, with the AI glowing pink
t1 = fig.text(0.06, 0.955, "What ", color=INK, fontproperties=F(poster.SERIF, 58), va="top")
fig.canvas.draw()
x_ai = 0.06 + t1.get_window_extent().width / fig.bbox.width
t2 = fig.text(x_ai, 0.955, HIGHLIGHT, color=PINK, fontproperties=F(poster.SERIF, 58), va="top")
fig.canvas.draw()
x_dr = x_ai + t2.get_window_extent().width / fig.bbox.width
fig.text(x_dr, 0.955, " Drinks", color=INK, fontproperties=F(poster.SERIF, 58), va="top")
fig.text(0.06, 0.872, f"{STATS['stressed']} of {N} AI campuses sit in basins under high water stress.\n"
                      f"Only {STATS['dropping']} sit over groundwater that is dropping fast.",
         color=SOFT, fontproperties=F(poster.SANS, 16), va="top", linespacing=1.5)

# legend
lx, ly = 0.06, 0.185
fig.text(lx, ly, "WATER STRESS OF THE BASIN", color=SOFT, fontproperties=F(poster.SANS_BOLD, 10.5), va="bottom")
for i, cat in enumerate([0, 1, 2, 3, 4]):
    fig.add_artist(plt.Rectangle((lx + i * 0.075, ly - 0.03), 0.07, 0.018, transform=fig.transFigure, color=WATER[cat]))
    fig.text(lx + i * 0.075, ly - 0.036, WATER_LABEL[cat], color=SOFT, fontproperties=F(poster.SANS, 9), va="top")
for i, (lab, kw) in enumerate([("AI campus, running", dict(color=RUNNING)), ("AI campus, coming", dict(color=COMING)),
                               ("Groundwater dropping fast", dict(facecolor="none", edgecolor=INK, lw=1.6, ls=(0, (2.5, 1.5))))]):
    y = ly - i * 0.024
    fig.add_artist(plt.matplotlib.patches.Circle((0.52 + 0.0065, y - 0.004), 0.0065, transform=fig.transFigure, **kw))
    fig.text(0.545, y - 0.004, lab, color=INK, fontproperties=F(poster.SANS, 12), va="center")

poster.credit(fig, "WRI Aqueduct 4.0 · Epoch AI Frontier Data Centers · US Census\n"
                   "Basins are where each campus sits, not its exact water source.",
              x=0.06, y=0.025, color=SOFT, path=poster.SANS, name_path=poster.SANS_BOLD, name_color=INK)
plt.show()

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE,
             subtitle=f"{STATS['stressed']} of {N} AI campuses in high water stress, {STATS['dropping']} over fast-dropping groundwater",
             series="ai", study=STUDY, tool="Python", notebook=NOTEBOOK, credits=CREDITS)
ai.update_master(campus.rename(columns={"basin_gtd": "gtd_cat"}), ["gtd_cat"],
                 about="Aqueduct groundwater table decline category of the basin", source="WRI Aqueduct 4.0", day=DAY)